In [1]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

# 假设原始数据参数
fs = 2000  # 采样率必须大于 900Hz 才能支持 450Hz 的带通截止频率
duration = 5  # 模拟 5 秒的信号
t = np.linspace(0, duration, fs * duration, endpoint=False)
# 随机生成股直肌原始 sEMG 信号（模拟高斯白噪声并赋予一定的振幅波动）
raw_emg = np.random.randn(len(t)) * np.sin(2 * np.pi * 0.5 * t)

# 第一步：20-450Hz 带通滤波
nyq = 0.5 * fs
low = 20 / nyq
high = 450 / nyq
b_band, a_band = signal.butter(4, [low, high], btype='bandpass')
emg_bp = signal.filtfilt(b_band, a_band, raw_emg)

# 第二步：50Hz 陷波滤波器剔除工频干扰
freq = 50.0
Q = 30.0  # 品质因数
b_notch, a_notch = signal.iirnotch(freq, Q, fs)
emg_notch = signal.filtfilt(b_notch, a_notch, emg_bp)

# 第三步：全波整流
emg_rectified = np.abs(emg_notch)

# 第四步：低通滤波提取线性包络（通常截止频率设为 6-10Hz）
cutoff_env = 10 / nyq
b_lp, a_lp = signal.butter(4, cutoff_env, btype='lowpass')
emg_envelope = signal.filtfilt(b_lp, a_lp, emg_rectified)

# 第五步：假设最大自发收缩 (MVC) 进行标准化
assumed_mvc = np.max(emg_envelope) * 1.5  # 假设一次 MVC 测试获得的峰值比当前动作峰值高
emg_normalized = (emg_envelope / assumed_mvc) * 100  # 转化为 MVC 的百分比 (%)

# 打印最终数组的前几个标准化数值作为验证
print("Normalized EMG Output (first 10 samples in % MVC):")
print(np.round(emg_normalized[:10], 2))

Normalized EMG Output (first 10 samples in % MVC):
[1.99 2.04 2.09 2.14 2.2  2.25 2.3  2.36 2.41 2.47]
